# RAG In-Class Exercise

> **Goal:** Use small experiments to understand how individual RAG design choices affect retrieval and generation.

This notebook is designed to be used **after the corresponding PPT explanations**.

## How to Use This Notebook

| Cell type | What you should do | Visual cue |
|---|---|---|
| ▶️ Run-only | Run the cell as-is. Do not edit it. | Blue callout |
| ✏️ Mini Lab | Edit the marked parameters or question, then rerun the cell. | Yellow callout |
| 🔍 Observe | Inspect the output and discuss what changed. | Green callout |

## Notebook Roadmap

1. Environment Setup
2. Configuration
3. Load Documents
4. In-Class Exercise 1 — Chunk Size
   - 4.1 Mini Lab 1 — Compare Chunk Sizes ✏️
5. In-Class Exercise 2 — Embedding and Similarity
   - 5.1 Observe Similarity Directly
   - 5.2 A Useful Mental Model
   - 5.3 Connect Embeddings to a Vector Index
   - 5.4 What Has Happened?
6. In-Class Exercise 3 — Retrieval and `top_k`
   - 6.1 Inspect Retrieval
   - 6.2 Build the Complete RAG Query Function
   - 6.3 Mini Lab 2 — Change `top_k` ✏️
7. In-Class Exercise 4 — Inspect the Prompt
8. In-Class Exercise 5 — RAG vs. LLM-only

# 1. Environment Setup

Please ensure that Python 3.9 or a later version is installed on your machine. We recommend using Miniconda to manage virtual environments.

### Core Tools

- **Python / Jupyter Notebook**: Environment for instruction and experiments
- **LlamaIndex**: Orchestration framework for the RAG pipeline
- **Chroma**: Local vector database
- **Qwen Embedding**: Maps text to vectors
- **Qwen LLM**: Generates answers based on retrieved context
- **PyPDF**: PDF text parsing

<div style="padding:14px 16px; border-left:8px solid #1f77b4; background:#f0f7ff; border-radius:8px;">
<b>▶️ Run-only cell</b><br>
Run the cells in this section as-is. No edits are needed.
</div>

In [ ]:
# test python and environment
import sys
import platform
from pathlib import Path

print("=" * 60)
print("Environment Check")
print("=" * 60)

print(f"Python       : {sys.version}")
print(f"Executable   : {sys.executable}")
print(f"Platform     : {platform.platform()}")
print(f"Working dir  : {Path.cwd().resolve()}")

assert sys.version_info >= (3, 9), "Python >= 3.9 is required."

print("\n✅ Python version check passed.")


## 1.1 Dependency Check

Prepare the environment once before class:

```powershell
conda activate myenv   # Replace `myenv` with your virtual environment name
```

> Keeping installation outside the notebook avoids package-version changes, kernel mismatch, and unexpected restarts during class. The notebook only checks that the required packages can be imported.

In [ ]:
# check package
import importlib

packages = {
    "llama_index": "LlamaIndex",
    "chromadb": "Chroma",
    "openai": "OpenAI SDK",
    "pypdf": "PyPDF",
    "dotenv": "python-dotenv",
    "pandas": "Pandas",
    "sklearn": "scikit-learn",
}

print("=" * 60)
print("Package Check")
print("=" * 60)

for module, name in packages.items():
    try:
        importlib.import_module(module)
        print(f"✅ {name}")
    except ImportError as e:
        print(f"❌ {name}: {e}")


# 2. Configuration

Here, the **provider, model, and API endpoint** are managed centrally.

This way, when switching from:

```text
Qwen → OpenAI / Azure OpenAI / other provider
```

...you only need to modify the configuration rather than rewriting the entire RAG pipeline.

### Recommended `.env`

```text
DASHSCOPE_API_KEY=your_api_key
DASHSCOPE_BASE_URL=https://dashscope.aliyuncs.com/compatible-mode/v1
LLM_MODEL=qwen3.7-flash
EMBEDDING_MODEL=qwen3.7-text-embedding
```

> Do not hardcode your actual API key into the notebook, and do not commit the `.env` file to GitHub.

> **Regional endpoint note:** Model Studio API keys are region-bound. If you use a region-specific Base URL, use an API key created for that same region.

<div style="padding:14px 16px; border-left:8px solid #1f77b4; background:#f0f7ff; border-radius:8px;">
<b>▶️ Run-only cell</b><br>
Run the cells in this section as-is. No edits are needed.
</div>

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY") or os.getenv("dashscope_api_key")
DASHSCOPE_BASE_URL = (
    os.getenv("DASHSCOPE_BASE_URL")
    or os.getenv("dashscope_base_url")
    or "https://dashscope.aliyuncs.com/compatible-mode/v1"
)

LLM_MODEL = os.getenv("LLM_MODEL", "qwen3.7-flash")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "qwen3.7-text-embedding")

print("=" * 60)
print("Model Configuration")
print("=" * 60)

print(f"LLM model       : {LLM_MODEL}")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Base URL        : {DASHSCOPE_BASE_URL}")

if DASHSCOPE_API_KEY:
    print("API key         : ✅ loaded")
else:
    print("API key         : ❌ missing")

print("\n✅ Model configuration successful.")

print("\n✅ Section 2 complete: model configuration loaded.")


## 2.1 API Connectivity Test

API connection test: verify that the API is available, the URL is correct, and the model selection is correct.

Two test scripts:

1. LLM can answer questions (chat completion model);
2. Embedding API can convert text into vectors (text-embedding model).

In [ ]:
from openai import OpenAI

assert DASHSCOPE_API_KEY, (
    "DASHSCOPE_API_KEY not found. "
    "Please configure your .env file first."
)

client = OpenAI(
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
)

# ---- Test LLM ----
response = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "user", "content": "Explain what RAG is in a single sentence."}
    ],
)

print("LLM test:")
print(response.choices[0].message.content)

# ---- Test Embedding ----
embedding_response = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input="This is an embedding test sentence."
)

embedding = embedding_response.data[0].embedding

print("\nEmbedding test:")
print(f"Vector dimension = {len(embedding)}")
print(f"First 10 values  = {embedding[:10]}")


# 3. Load Documents

Now we enter the RAG pipeline.

## 3.1 Understand the `data/` Folders

```text
data/
├── raw/
│   ├── document_01.pdf
│   ├── document_02.pdf
│   └── ...
├── processed/
│   └── optional cleaned/preprocessed files
├── eval/
│   └── optional evaluation dataset (JSON/CSV/JSONL)
└── vector_db/
    └── automatically generated Chroma database
```

### `raw/` — original knowledge base

Put the **original PDF files** here. This notebook reads the PDFs from this folder.

### `processed/` — processed data

This folder is reserved for data after preprocessing, such as cleaning, OCR, format correction, or other transformations.

> In this notebook, `processed/` is **not used yet**. It can remain empty.

### `eval/` — evaluation data

This folder is reserved for evaluation questions and reference answers/sources, for example:

```json
[
  {
    "question": "What is ...?",
    "gold_answer": "...",
    "gold_pages": [9]
  }
]
```

> For teaching simplicity, the current evaluation set is defined directly in Python later in the notebook. `data/eval/` is reserved for a future external JSON/CSV/JSONL evaluation set.

### `vector_db/` — vector database

Students do **not** put PDFs here. Chroma automatically stores the local vector database here.

<div style="padding:14px 16px; border-left:8px solid #1f77b4; background:#f0f7ff; border-radius:8px;">
<b>▶️ Run-only cell</b><br>
Run the cells in this section as-is. No edits are needed.
</div>

In [ ]:
from pathlib import Path

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EVAL_DIR = DATA_DIR / "eval"
VECTOR_DB_DIR = DATA_DIR / "vector_db"

for folder in [RAW_DIR, PROCESSED_DIR, EVAL_DIR, VECTOR_DB_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

pdf_files = sorted(RAW_DIR.glob("*.pdf"))

print("=" * 60)
print("Section 3 — PDF Inventory")
print("=" * 60)

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found in {RAW_DIR.resolve()}. "
        "Please copy the course PDFs into data/raw/."
    )

for i, pdf in enumerate(pdf_files, start=1):
    print(f"{i:>2}. {pdf.name}  ({pdf.stat().st_size / 1024:.1f} KB)")

print(f"\nTotal PDFs: {len(pdf_files)}")
print("\n✅ Section 3 complete: data folders and PDF inventory ready.")


In [ ]:
from llama_index.core import SimpleDirectoryReader

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found in {RAW_DIR.resolve()}.\n"
        "Please copy your course PDFs into data/raw/."
    )

reader = SimpleDirectoryReader(
    input_dir=str(RAW_DIR),
    # You can also set `required_exts` to load only files with specific extensions.
    required_exts=[".pdf"],
    recursive=True,
)

documents = reader.load_data(show_progress=True)

print(f"\nLoaded document units: {len(documents)}")


## 3.2 Inspect the Loaded Documents

For PDFs, each `Document` often represents one page or one parsed unit. It is not the final retrieval chunk.

At this stage, document-level metadata such as the file name and page number is preserved. This metadata can later support source citations and provenance tracking.

In [ ]:
from IPython.display import display

# Inspect the first five document units
for i, doc in enumerate(documents[:5], start=1):
    print("=" * 80)
    print(f"Document unit #{i}")
    print(f"Metadata: {doc.metadata}")
    print(f"Text preview:\n{doc.text[:700]}")


In [ ]:
# Run a simple data quality check
empty_docs = [i for i, doc in enumerate(documents) if not doc.text.strip()]
text_lengths = [len(doc.text) for doc in documents if doc.text.strip()]

print(f"Non-empty units : {len(text_lengths)}")
print(f"Empty units     : {len(empty_docs)}")

if text_lengths:
    print(f"Min chars      : {min(text_lengths):,}")
    print(f"Median chars   : {sorted(text_lengths)[len(text_lengths)//2]:,}")
    print(f"Max chars      : {max(text_lengths):,}")


# 4. In-Class Exercise 1 — Chunk Size

Chunking determines the unit of knowledge that the retriever can return.

Before running the code, predict how smaller or larger chunks might affect context, retrieval precision, and the number of chunks. We will inspect a sample chunk and the length distribution. A histogram describes chunk sizes, but cannot tell us whether semantic boundaries are good.

Validate chunking by checking the content, retrieval results, and answer quality. The current setting (`chunk_size = 512`, `chunk_overlap = 100`) is a starting point, not a universal optimum.

<div style="padding:14px 16px; border-left:8px solid #1f77b4; background:#f0f7ff; border-radius:8px;">
<b>▶️ Run-only cell</b><br>
Run this cell as-is. The editable version is in Mini Lab 1 below.
</div>

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

CHUNK_SIZE = 512
CHUNK_OVERLAP = 100

splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

nodes = splitter.get_nodes_from_documents(documents)

print("=" * 60)
print("Chunking Result")
print("=" * 60)

print(f"Original document units : {len(documents)}")
print(f"Generated chunks        : {len(nodes)}")
print(f"Chunk size              : {CHUNK_SIZE}")
print(f"Chunk overlap           : {CHUNK_OVERLAP}")

print("\n✅ Section 4 complete: chunks generated.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if not nodes:
    raise ValueError("No chunks were generated. Check the loaded documents.")

sample_node = nodes[min(3, len(nodes) - 1)]
print("=" * 80)
print("Sample Chunk")
print("=" * 80)
print(f"Node ID : {sample_node.node_id}")
print(f"Metadata: {sample_node.metadata}")
print(f"\nText:\n{sample_node.text}")

chunk_lengths = np.array([len(node.text) for node in nodes])
print("\nChunk length statistics:")
print(pd.Series(chunk_lengths).describe())

plt.figure(figsize=(8, 4))
plt.hist(chunk_lengths, bins=30)
plt.xlabel("Characters in chunk")
plt.ylabel("Number of chunks")
plt.title("Chunk length distribution")
plt.show()

print("The histogram describes chunk sizes, not semantic quality.")
print("Also check chunk content, retrieval results, and answer quality.")


<div style="padding:16px; border-left:8px solid #f5a623; background:#fff8e1; border-radius:8px;">
<h2 style="margin-top:0;">✏️ Mini Lab 1 — Compare Chunk Sizes</h2>

**Editable cell:** Change the chunk settings in the code cell below and rerun it.

**Goal:** See how `chunk_size` and `chunk_overlap` change chunk boundaries and the amount of context in each retrieved unit.

**Compare:**

- A: `chunk_size = 300`,  `overlap = 50`
- B: `chunk_size = 600`,  `overlap = 100`
- C: `chunk_size = 1000`, `overlap = 150`

**Ask yourself:**

- Does a smaller chunk contain a more focused piece of information?
- Does a larger chunk preserve more context?
- Which setting is more likely to retrieve the exact information we need?
- What information might be lost when a chunk is too small?
- What noise might be added when a chunk is too large?

> **Key idea:** `chunk_size` controls the amount of context that can be retrieved as one unit.
</div>

In [ ]:
# ============================================================
# ✏️ MINI LAB 1 — EDIT THE CHUNK SETTINGS BELOW
# ============================================================
chunk_settings = [
    (300, 50),
    (600, 100),
    (1000, 150),
]
# ============================================================
# ▶️ RUN THE REST OF THIS CELL WITHOUT EDITING
# ============================================================

from IPython.display import display, HTML
import html
import numpy as np
import pandas as pd

# Use a relatively long source unit so the effect of chunk size is visible.
source_doc = max(documents, key=lambda d: len(d.text))

comparison_rows = []

for size, overlap in chunk_settings:
    splitter_tmp = SentenceSplitter(
        chunk_size=size,
        chunk_overlap=overlap,
    )
    nodes_tmp = splitter_tmp.get_nodes_from_documents([source_doc])

    lengths = np.array([len(n.text) for n in nodes_tmp])
    comparison_rows.append({
        "chunk_size": size,
        "overlap": overlap,
        "num_chunks": len(nodes_tmp),
        "avg_chars": round(lengths.mean(), 1),
        "median_chars": round(np.median(lengths), 1),
    })

    # Show the first few actual chunks so students can see the boundaries.
    cards = []
    for j, node in enumerate(nodes_tmp[:3], start=1):
        text = html.escape(node.text).replace("\n", "<br>")
        cards.append(f"""
        <div style="margin:10px 0; padding:12px; border:1px solid #cccccc; border-radius:8px;">
            <b>Chunk {j}</b> &nbsp; <span style="color:#666;">{len(node.text)} characters</span>
            <div style="margin-top:8px; line-height:1.5;">{text}</div>
        </div>
        """)

    display(HTML(f"""
    <div style="margin:16px 0; padding:14px; border:2px solid #888; border-radius:10px;">
        <h3 style="margin-top:0;">chunk_size = {size}, overlap = {overlap}</h3>
        <p><b>{len(nodes_tmp)} chunks</b> generated from the same source unit.</p>
        {''.join(cards)}
        {f'<p style="color:#666;">Showing the first 3 chunks only.</p>' if len(nodes_tmp) > 3 else ''}
    </div>
    """))

print("Summary:")
display(pd.DataFrame(comparison_rows))

print("\nWhat should you notice?")
print("• Smaller chunks: more focused, but a complete fact may be split across chunks.")
print("• Larger chunks: more context, but each retrieved chunk may contain more unrelated information.")
print("• The table describes the quantity; the chunk previews show the actual semantic difference.")
print("\nTeaching rule: do not choose a setting from chunk count alone.")
print("Use retrieval and answer quality to validate the choice.")
print("\n✅ Mini Lab 1 complete: chunk boundaries and trade-offs inspected.")


# 5. In-Class Exercise 2 — Embedding and Similarity

Now ask:

> **How does the system decide whether two pieces of text are similar?**

An embedding model converts text into vectors:

```text
Text
  ↓
Embedding model
  ↓
[0.013, -0.291, 0.044, ..., 0.182]
```

The individual numbers are not important for us.

The key idea is:

> **Semantically similar texts tend to be closer together in vector space.**

### Why `embed_batch_size=20`?

The embedding API accepts at most 20 texts in a single request. LlamaIndex therefore sends the chunks in batches of 20 or fewer.

<div style="padding:14px 16px; border-left:8px solid #1f77b4; background:#f0f7ff; border-radius:8px;">
<b>▶️ Run-only cell</b><br>
Run the cells in this section as-is. No edits are needed.
</div>

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding

embed_model = OpenAIEmbedding(
    model_name=EMBEDDING_MODEL,
    api_key=DASHSCOPE_API_KEY,
    api_base=DASHSCOPE_BASE_URL,
    embed_batch_size=20,
)

test_texts = [
    "The building's primary function is for office use.",
    "This is an office building.",
    "The weather is nice today.",
]

test_embeddings = [
    embed_model.get_text_embedding(text)
    for text in test_texts
]

print(f"Embedding dimension: {len(test_embeddings[0])}")

for text, vec in zip(test_texts, test_embeddings):
    print("\nText:", text)
    print("First 8 dimensions:", vec[:8])

print("\n✅ Section 5 complete: embedding experiment finished.")


## 5.1 Observe Similarity Directly

Before using a vector database, we can calculate cosine similarity ourselves.

This makes the retrieval mechanism easier to understand:

```text
text → embedding → similarity score
```

<div style="padding:14px 16px; border-left:8px solid #1f77b4; background:#f0f7ff; border-radius:8px;">
<b>▶️ Run-only cell</b><br>
Run the cells in this subsection as-is. No edits are needed.
</div>

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

matrix = np.array(test_embeddings)

similarity_matrix = cosine_similarity(matrix)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=["Text 1", "Text 2", "Text 3"],
    columns=["Text 1", "Text 2", "Text 3"],
)

display(similarity_df.round(3))


In [ ]:
query = "What functions does this office building have?"
query_embedding = np.array(
    embed_model.get_text_embedding(query)
).reshape(1, -1)

scores = cosine_similarity(query_embedding, matrix)[0]

result = pd.DataFrame({
    "text": test_texts,
    "similarity": scores,
}).sort_values("similarity", ascending=False)

display(result.round({"similarity": 3}))


## 5.2 A Useful Mental Model

A vector database efficiently stores and searches embedding vectors. It does not “think”; it primarily:

```text
store vectors
      +
search nearest vectors
      +
return associated text / metadata
```

## 5.3 Connect Embeddings to a Vector Index

The previous exercise showed how embeddings represent text and how similarity can be calculated.

Now we let **Chroma** store the vectors and **LlamaIndex** connect them to the chunks.

```text
Documents
   ↓
Chunks
   ↓
Embeddings
   ↓
Vector Database
```

This is the bridge from the indexing exercises to retrieval.

<div style="padding:14px 16px; border-left:8px solid #1f77b4; background:#f0f7ff; border-radius:8px;">
<b>▶️ Run-only cell</b><br>
Run the cells in this subsection as-is. No edits are needed.
</div>

In [ ]:
import chromadb
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore

COLLECTION_NAME = "rag_course_demo"

chroma_client = chromadb.PersistentClient(
    path=str(VECTOR_DB_DIR)
)

# Rebuild collection for a clean classroom run
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.get_or_create_collection(
    COLLECTION_NAME
)

vector_store = ChromaVectorStore(
    chroma_collection=collection
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

print(f"Chroma collection: {COLLECTION_NAME}")
print(f"Persistent path  : {VECTOR_DB_DIR.resolve()}")

print("\n✅ Section 5.3 complete: vector index created.")


In [ ]:
# Build the index from the chunks we explicitly created above.
index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=True,
)

print("\n✅ Vector index created.")
print(f"Indexed chunks: {len(nodes)}")


## 5.4 What Has Happened?

The indexing phase is complete:

```text
PDF
 →
Documents
 →
Chunks
 →
Embeddings
 →
Chroma
```

The knowledge base is now built, but this is not yet a complete RAG query. Next comes:

```text
Query → Retrieval → Augmentation → Generation
```

# 6. In-Class Exercise 3 — Retrieval and `top_k`

The retriever takes a question and returns the most similar chunks:

```text
Question
   ↓
Question embedding
   ↓
Similarity search
   ↓
Top-k chunks
```

Remember:

> **The retriever does not answer the question. It only selects context.**

<div style="padding:14px 16px; border-left:8px solid #1f77b4; background:#f0f7ff; border-radius:8px;">
<b>▶️ Run-only cell</b><br>
Run the cells in this section as-is. The editable version is in Mini Lab 2 below.
</div>

In [ ]:
TOP_K = 4

retriever = index.as_retriever(
    similarity_top_k=TOP_K
)

print(f"Retriever configured with top_k = {TOP_K}")

print("\n✅ Section 6.1 complete: retriever is ready.")


## 6.1 Inspect Retrieval

Before comparing `top_k` values, let us look at what a single retrieval pass actually returns.

<div style="padding:14px 16px; border-left:8px solid #1f77b4; background:#f0f7ff; border-radius:8px;">
<b>▶️ Run-only cell</b><br>
Run the cells in this subsection as-is. No edits are needed.
</div>

In [ ]:
import re
import html
from IPython.display import display, HTML

def get_file_name(node):
    metadata = node.metadata or {}
    return (
        metadata.get("file_name")
        or metadata.get("filename")
        or metadata.get("file_path")
        or "Unknown"
    )

def get_page_number(node):
    metadata = node.metadata or {}
    for key in ["page_label", "page_number", "page"]:
        value = metadata.get(key)
        if value is not None:
            try:
                return int(value)
            except (TypeError, ValueError):
                return value
    return None

def highlight_query_terms(text, query):
    """Highlight informative query terms in red."""
    stopwords = {
        "what", "is", "the", "a", "an", "of", "for", "on", "in", "to",
        "and", "each", "type", "be", "at", "as", "how", "many", "does",
        "will", "within", "according", "this", "document"
    }

    terms = [
        word.lower()
        for word in re.findall(r"\b[a-zA-Z0-9-]+\b", query)
        if word.lower() not in stopwords
    ]
    terms = sorted(set(terms), key=len, reverse=True)

    highlighted = html.escape(text)

    for term in terms:
        pattern = re.compile(
            rf"(?<![\w-])({re.escape(term)})(?![\w-])",
            re.IGNORECASE
        )
        highlighted = pattern.sub(
            r'<span style="color:#b00000; font-weight:bold;">\1</span>',
            highlighted
        )
    return highlighted

def inspect_retrieval(question, top_k=TOP_K, retrieved_nodes=None):

    # Run retrieval if results were not provided.
    if retrieved_nodes is None:
        retriever = index.as_retriever(
            similarity_top_k=top_k
        )
        retrieved_nodes = retriever.retrieve(question)

    print("=" * 80)
    print("RETRIEVED CHUNKS")
    print("=" * 80)

    for rank, item in enumerate(retrieved_nodes, start=1):

        node = item.node
        score = item.score

        file_name = get_file_name(node)
        page = get_page_number(node)

        print(f"\nRank {rank}")

        if score is not None:
            print(f"Score: {score:.4f}")
        else:
            print("Score: N/A")

        print(f"File : {file_name}")
        print(f"Page : {page}")

        print("\nChunk:")

        highlighted_text = highlight_query_terms(
            node.text,
            question
        )

        display(HTML(highlighted_text))

print("-" * 80)
print("\n✅ Section 6.1 setup complete: retrieval inspection function is ready.")


In [ ]:
question = (
    "What is the minimum sampling rate for on-site quality audit checks on each type of modular unit delivered to the building site?"
)

inspect_retrieval(question)

print("\n✅ Section 6.1 complete: retrieval results inspected.")


<div style="padding:14px 16px; border-left:8px solid #2ca02c; background:#f2fff2; border-radius:8px;">
<b>🔍 Observe & Discuss — What Makes Retrieval Good?</b><br><br>

Look at the retrieved chunks and ask:

- Is the highest-ranked chunk actually useful?
- Are all returned chunks relevant?
- Does a lower-ranked chunk contain an important fact?
- Could increasing `top_k` add useful context, or just add noise?

This is why retrieval quality is an important part of RAG.
</div>

## 6.2 Build the Complete RAG Query Function

Now connect the modules from the previous sections:

```text
Question
   ↓
Retriever
   ↓
Top-k chunks
   ↓
Prompt augmentation
   ↓
Qwen LLM
   ↓
Answer
```

We will implement the pipeline explicitly once because assembling it step by step is easier to follow in a teaching setting than calling `query_engine.query()` directly.

<div style="padding:14px 16px; border-left:8px solid #1f77b4; background:#f0f7ff; border-radius:8px;">
<b>▶️ Run-only cell</b><br>
Run the cells in this subsection as-is. No edits are needed.
</div>

In [ ]:
from llama_index.core.llms import ChatMessage
from llama_index.llms.openai_like import OpenAILike

llm = OpenAILike(
    model=LLM_MODEL,
    api_key=DASHSCOPE_API_KEY,
    api_base=DASHSCOPE_BASE_URL,
    temperature=0.2,
    is_chat_model=True,
)

print(f"LLM ready: {LLM_MODEL}")

print("\n✅ Section 6.2 complete: LLM is ready.")


In [ ]:
SYSTEM_PROMPT = """You are a rigorous knowledge base question-answering assistant.

Please follow these rules:
1. Answer only based on the provided Context.
2. If the Context does not contain sufficient information to answer the question, explicitly state: “The knowledge base does not contain sufficient information.”
3. Do not add or infer facts that are not explicitly provided in the Context.
4. Keep your answers concise and clear whenever possible.
5. At the end of your answer, provide the names of the source files used.
"""

def build_context(retrieved_nodes):
    context_blocks = []

    for i, item in enumerate(retrieved_nodes, start=1):
        node = item.node
        file_name = (
            node.metadata.get("file_name")
            or node.metadata.get("filename")
            or "unknown_file"
        )

        page = get_page_number(node)
        page_label = f", page {page}" if page is not None else ""

        context_blocks.append(
            f"[Source {i}] {file_name}{page_label}\n"
            f"{node.text}"
        )

    return "\n\n".join(context_blocks)


def build_rag_messages(question, retrieved_nodes):
    context = build_context(retrieved_nodes)

    user_prompt = f"""Please answer the following question.

Question:
{question}

Context:
{context}

Strictly base your answer on the provided Context.
"""

    return [
        ChatMessage(role="system", content=SYSTEM_PROMPT),
        ChatMessage(role="user", content=user_prompt),
    ]

def rag_query(question, top_k=TOP_K, verbose=True):

    # --------------------------------------------------
    # 1. Retrieval
    # --------------------------------------------------
    retriever = index.as_retriever(
        similarity_top_k=top_k
    )

    retrieved_nodes = retriever.retrieve(question)

    # --------------------------------------------------
    # 2. Build context
    # --------------------------------------------------
    messages = build_rag_messages(
        question,
        retrieved_nodes
    )

    # --------------------------------------------------
    # 3. Generate answer
    # --------------------------------------------------
    response = llm.chat(messages)

    answer = response.message.content

    # --------------------------------------------------
    # 4. Display
    # --------------------------------------------------
    if verbose:

        print("=" * 80)
        print("QUESTION")
        print("=" * 80)
        print(question)

        print("\nANSWER")
        print("=" * 80)
        print(answer)

        print("\nRETRIEVED CHUNKS")
        print("=" * 80)

        # Reuse retrieved nodes to avoid running retrieval again.
        inspect_retrieval(
            question,
            retrieved_nodes=retrieved_nodes
        )

    # --------------------------------------------------
    # 5. Collect source information
    # --------------------------------------------------
    sources = []

    for rank, item in enumerate(retrieved_nodes, start=1):

        node = item.node

        sources.append({
            "rank": rank,
            "score": (
                float(item.score)
                if item.score is not None
                else None
            ),
            "file": get_file_name(node),
            "page": get_page_number(node),
            "node_id": node.node_id,
        })

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "retrieved_nodes": retrieved_nodes,
    }

print("✅ Section 6.2 setup complete: RAG query function is ready.")


## 6.3 Mini Lab 2 — Change `top_k`

<div style="padding:16px; border-left:8px solid #f5a623; background:#fff8e1; border-radius:8px;">
<h2 style="margin-top:0;">✏️ Mini Lab 2 — Change <code>top_k</code></h2>

**Editable cell:** Change `topk_values` and optionally the question in the code cell below, then rerun it.

**Goal:** Observe how the number of retrieved chunks affects context quality and the final answer.

**Try:**

- `topk_values = [2, 4, 8]`
- `topk_values = [1, 3, 6, 10]`

**Look for:**

- changes in ranking;
- useful vs. irrelevant chunks;
- whether more context helps or adds noise;
- whether the final answer becomes more complete or less focused.

> The output highlights informative query terms in **red** so you can quickly connect the question with the retrieved text.
</div>

In [ ]:
# ============================================================
# ✏️ MINI LAB 2 — EDIT THE VALUES BELOW
# ============================================================
topk_values = [2, 4, 8]

topkquestion = (
    "What is the minimum sampling rate for on-site quality audit checks on each type of modular unit delivered to the building site?"
)
# ============================================================
# ▶️ RUN THE REST OF THIS CELL WITHOUT EDITING
# ============================================================

from llama_index.core.llms import ChatMessage

for k in topk_values:
    print(f"\n{'=' * 80}\nTOP_K = {k}\n{'=' * 80}")

    # Retrieve the actual nodes so students can inspect the content.
    retriever_tmp = index.as_retriever(similarity_top_k=k)
    retrieved_nodes = retriever_tmp.retrieve(topkquestion)

    print("\nRetrieved chunks:")
    inspect_retrieval(
        topkquestion,
        top_k=k,
        retrieved_nodes=retrieved_nodes
    )

    print("\nRAG answer:")
    result = rag_query(
        topkquestion,
        top_k=k,
        verbose=False
    )
    print(result["answer"])

print("\n✅ Mini Lab 2 complete: top-k values inspected.")


# 7. In-Class Exercise 4 — Inspect the Prompt

RAG does not send the original PDF directly to the LLM.

Instead:

```text
PDF
 ↓
Chunk
 ↓
Retrieved chunks
 ↓
Prompt + context
 ↓
LLM
```

Inspect the final prompt and identify:

1. the system instructions;
2. the user question;
3. the retrieved context.

Ask:

> **What exactly does the LLM see when it generates the answer?**

<div style="padding:14px 16px; border-left:8px solid #1f77b4; background:#f0f7ff; border-radius:8px;">
<b>▶️ Run-only cell</b><br>
Run the cells in this section as-is. No edits are needed.
</div>

In [ ]:
def show_rag_prompt(question, top_k=4):
    retrieved_nodes = index.as_retriever(
        similarity_top_k=top_k
    ).retrieve(question)

    messages = build_rag_messages(
        question,
        retrieved_nodes
    )

    print("=" * 80)
    print("SYSTEM PROMPT")
    print("=" * 80)
    print(messages[0].content)

    print("\n" + "=" * 80)
    print("AUGMENTED USER PROMPT")
    print("=" * 80)
    print(messages[1].content)

show_rag_prompt(
    "What quality assurance certification can be used for a prefabrication factory under Appendix B?",
    top_k=4
)


# 8. In-Class Exercise 5 — RAG vs. LLM-only

Now compare two systems using the same LLM:

### LLM-only
```text
Question → LLM → Answer
```

### RAG
```text
Question → Retrieval → Context → LLM → Answer
```

The question is not simply whether the LLM can answer.

Instead ask:

> **What changes when the LLM is given relevant external knowledge from the knowledge base?**

<div style="padding:14px 16px; border-left:8px solid #1f77b4; background:#f0f7ff; border-radius:8px;">
<b>▶️ Run-only cell</b><br>
Run the cells in this section as-is. No edits are needed.
</div>

In [ ]:
def llm_only(question):
    messages = [
        ChatMessage(
            role="system",
            content="Answer the user's question directly. If you are uncertain, explicitly state that you are uncertain."
        ),
        ChatMessage(
            role="user",
            content=question
        ),
    ]

    response = llm.chat(messages)
    return response.message.content

comparison_question = (
    "Within how many days will the Building Department determine a pre-submission enquiry under ADV-36?"
)

print("=" * 80)
print("LLM ONLY")
print("=" * 80)
print(llm_only(comparison_question))

print("\n" + "=" * 80)
print("RAG")
print("=" * 80)

result = rag_query(
    comparison_question,
    top_k=4
)
